In [1]:
from gurobipy import Model, GRB
import pandas as pd

# Create a new model
model = Model("Par_Inc_Profit_Maximization")

# Decision variables
S = model.addVar(vtype=GRB.CONTINUOUS, name="Standard_Bags")  # Standard golf bags
D = model.addVar(vtype=GRB.CONTINUOUS, name="Deluxe_Bags")    # Deluxe golf bags

# Objective function: Maximize profit
model.setObjective(10 * S + 9 * D, GRB.MAXIMIZE)

# Constraints
# Stock difference constraint
model.addConstr(S - D <= 500, "Stock_Difference")

# Cutting and dyeing constraint
model.addConstr(7 * S + 10 * D <= 6300, "Cutting_and_Dyeing")

# Sewing constraint
model.addConstr(3 * S + 5 * D <= 3600, "Sewing")

# Finishing constraint
model.addConstr(3 * S + 2 * D <= 2124, "Finishing")

# Inspection and packaging constraint
model.addConstr(2 * S + 5 * D <= 2700, "Inspection_and_Packaging")

# Minimum production constraint
model.addConstr(S + D >= 400, "Minimum_Production")



# Non-negativity constraints are automatically enforced in Gurobi

# Optimize the model
model.optimize()
print(model.x)

Restricted license - for non-production use only - expires 2026-11-23
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) Ultra 7 165U, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 14 logical processors, using up to 14 threads

Optimize a model with 6 rows, 2 columns and 12 nonzeros
Model fingerprint: 0xc07238eb
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [9e+00, 1e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [4e+02, 6e+03]
Presolve time: 0.01s
Presolved: 6 rows, 2 columns, 12 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.9000000e+31   7.125000e+30   1.900000e+01      0s
       3    7.6680000e+03   0.000000e+00   0.000000e+00      0s

Solved in 3 iterations and 0.02 seconds (0.00 work units)
Optimal objective  7.668000000e+03
[540.0, 252.0]


In [3]:
var_names = []
for v in model.getVars():
  var_names.append(v.VarName)
  constr_names = []
for c in model.getConstrs():
  constr_names.append(c.ConstrName)
  var_columns = ['X', 'Obj','RC', 'SAObjLow', 'SAObjUp']
  constr_columns = ['Pi','RHS','Slack','SARHSLow','SARHSUp']
  SR_vars = pd.DataFrame(columns = var_columns, index = var_names, data = 0.0)
  SR_constrs = pd.DataFrame(columns = constr_columns, index = constr_names, data =
  0.0)
for v in model.getVars():
  for i in var_columns:
    SR_vars.loc[v.VarName, i] = v.getAttr(i)
    
for c in model.getConstrs():
  for i in constr_columns:
    SR_constrs.loc[c.ConstrName,i] = c.getAttr(i)


print(SR_vars)
print(SR_constrs)

                   X   Obj   RC  SAObjLow    SAObjUp
Standard_Bags  540.0  10.0  0.0  6.300000  13.500000
Deluxe_Bags    252.0   9.0  0.0  6.666667  14.285714
                              Pi     RHS  Slack  SARHSLow      SARHSUp
Stock_Difference          0.0000   500.0  212.0     288.0          inf
Cutting_and_Dyeing        0.4375  6300.0    0.0    5621.6  6823.636364
Sewing                    0.0000  3600.0  720.0    2880.0          inf
Finishing                 2.3125  2124.0    0.0    1740.0  2323.529412
Inspection_and_Packaging  0.0000  2700.0  360.0    2340.0          inf
Minimum_Production        0.0000   400.0 -392.0      -inf   792.000000
